In [ ]:
import os
import sys
import json
import pickle
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# --- COLAB SETUP ---
try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
    
    # Mount Drive
    drive.mount('/content/drive')
    
    DRIVE_BASE = "/content/drive/MyDrive/AML_Project"
    CODE_BASE = "/content/code"
    
    # Clone repository if needed
    REPO_URL = "https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git"
    BRANCH = "emre-second-step"
    
    if not os.path.exists(CODE_BASE):
        print(f"Cloning repository...")
        !git clone --recursive {REPO_URL} {CODE_BASE}
        os.chdir(CODE_BASE)
        !git fetch origin {BRANCH}
        !git checkout {BRANCH}
    else:
        os.chdir(CODE_BASE)
    
    if CODE_BASE not in sys.path:
        sys.path.append(CODE_BASE)
        
except ImportError:
    IN_COLAB = False
    print("Not running in Colab")
    DRIVE_BASE = "."
    CODE_BASE = os.getcwd()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# --- PATHS ---
if IN_COLAB:
    EGOVLP_FEATURE_DIR = os.path.join(DRIVE_BASE, "features/egovlp")
else:
    EGOVLP_FEATURE_DIR = "data/features/egovlp"

ANNOTATION_PATH = "annotations/annotation_json/step_annotations.json"
SPLIT_PATH = "er_annotations/recordings_combined_splits.json"

# Create output directory
OUTPUT_DIR = "extension_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"EgoVLP Features: {EGOVLP_FEATURE_DIR}")
print(f"Output Dir: {OUTPUT_DIR}")

## 1. Load Annotations and Splits

In [ ]:
# Load step annotations
with open(ANNOTATION_PATH, 'r') as f:
    step_annotations = json.load(f)
print(f"Loaded annotations for {len(step_annotations)} recordings")

# Load splits
with open(SPLIT_PATH, 'r') as f:
    splits = json.load(f)
print(f"Train: {len(splits['train'])}, Val: {len(splits['val'])}, Test: {len(splits['test'])}")

## 2. Feature Loading Utilities

In [ ]:
def load_egovlp_features(recording_id):
    """Load EgoVLP features for a recording."""
    # Local features are stored as .npz with a `video_features` array:
    #   data/features/egovlp/<recording_id>_360p_224.npz
    feature_path = os.path.join(EGOVLP_FEATURE_DIR, f"{recording_id}_360p_224.npz")
    if not os.path.exists(feature_path):
        return None
    data = np.load(feature_path)
    if "video_features" not in data:
        return None
    return data["video_features"]

def pool_features(features, start_time, end_time, fps=0.5):
    """
    Pool features within a time window using mean pooling.
    EgoVLP features in this repo are extracted approximately every ~2 seconds (fps=0.5).
    """
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    
    # Clamp to valid range
    start_frame = max(0, start_frame)
    end_frame = min(len(features), end_frame)
    
    if start_frame >= end_frame or start_frame >= len(features):
        return np.zeros(features.shape[1])
    
    return np.mean(features[start_frame:end_frame], axis=0)

# Test loading
test_rec = list(step_annotations.keys())[0]
test_features = load_egovlp_features(test_rec)
if test_features is not None:
    print(f"Test features shape: {test_features.shape}")
else:
    print("Could not load test features - check EGOVLP_FEATURE_DIR path")

## 3. Extract Step Embeddings using GT Boundaries

In [ ]:
def extract_step_embeddings_gt(annotations, feature_loader):
    """
    Extract step-level embeddings using ground truth boundaries.
    
    Returns:
        dict: {recording_id: {
            'step_embeddings': np.array of shape (num_steps, feature_dim),
            'step_labels': list of 0/1 (correct/error),
            'recipe_id': int recipe ID,
            'recipe_label': int 0/1 (recording-level error),
            'descriptions': list of step descriptions,
            'segments': list of (start, end) tuples,
            'step_ids': list of step IDs
        }}
    """
    processed_data = {}
    
    for rec_id, ann in tqdm(annotations.items(), desc="Extracting features"):
        features = feature_loader(rec_id)
        if features is None:
            continue
        
        step_embeddings = []
        step_labels = []
        step_descriptions = []
        step_segments = []
        step_ids = []
        
        for step in ann.get('steps', []):
            # CaptainCook4D step annotations use `start_time` / `end_time` in seconds.
            start_time = step.get('start_time', 0)
            end_time = step.get('end_time', start_time + 1)
            
            # Pool features for this step
            emb = pool_features(features, start_time, end_time)
            step_embeddings.append(emb)
            
            # Get label (0=correct, 1=error based on has_errors field)
            has_error = step.get('has_errors', False)
            label = 1 if has_error else 0
            step_labels.append(label)
            
            # Store metadata
            step_descriptions.append(step.get('description', ''))
            step_segments.append((start_time, end_time))
            step_ids.append(step.get('step_id', -1))
        
        if len(step_embeddings) > 0:
            recipe_id = int(str(rec_id).split('_')[0])
            recipe_label = int(any(step_labels))

            # Keep both legacy keys ('embeddings'/'labels') and the new keys
            # expected by extension_step2_verification_baseline.ipynb.
            step_embeddings_arr = np.array(step_embeddings)
            processed_data[rec_id] = {
                'step_embeddings': step_embeddings_arr,
                'step_labels': step_labels,
                'recipe_id': recipe_id,
                'recipe_label': recipe_label,
                'embeddings': step_embeddings_arr,
                'labels': step_labels,
                'descriptions': step_descriptions,
                'segments': step_segments,
                'step_ids': step_ids,
                'num_steps': len(step_embeddings)
            }
    
    return processed_data

# Extract embeddings
print("Extracting step embeddings using GT boundaries...")
processed_data = extract_step_embeddings_gt(step_annotations, load_egovlp_features)
print(f"Processed {len(processed_data)} recordings")

In [ ]:
# Statistics
total_steps = sum(d['num_steps'] for d in processed_data.values())
total_errors = sum(sum(d['labels']) for d in processed_data.values())
feature_dim = list(processed_data.values())[0]['embeddings'].shape[1]

print(f"\n=== Dataset Statistics ===")
print(f"Total recordings: {len(processed_data)}")
print(f"Total steps: {total_steps}")
print(f"Error steps: {total_errors} ({100*total_errors/total_steps:.1f}%)")
print(f"Correct steps: {total_steps - total_errors} ({100*(total_steps-total_errors)/total_steps:.1f}%)")
print(f"Feature dimension: {feature_dim}")

## 4. Save Output for Pipeline

In [ ]:
# Save processed data
output_data = {
    'data': processed_data,
    'splits': splits,
    'feature_dim': feature_dim,
    'method': 'ground_truth',
    'feature_type': 'egovlp'
}

# Save locally
local_output_path = os.path.join(OUTPUT_DIR, "step_embeddings_gt.pkl")
with open(local_output_path, 'wb') as f:
    pickle.dump(output_data, f)
print(f"Saved to: {local_output_path}")

# Save to Google Drive if in Colab
if IN_COLAB:
    drive_output_dir = os.path.join(DRIVE_BASE, "extension_data")
    os.makedirs(drive_output_dir, exist_ok=True)
    drive_output_path = os.path.join(drive_output_dir, "step_embeddings_gt.pkl")
    with open(drive_output_path, 'wb') as f:
        pickle.dump(output_data, f)
    print(f"Also saved to Drive: {drive_output_path}")

## 5. Verify Output

In [ ]:
# Verify the saved file
with open(local_output_path, 'rb') as f:
    loaded = pickle.load(f)

print("=== Verification ===")
print(f"Keys: {loaded.keys()}")
print(f"Recordings: {len(loaded['data'])}")
print(f"Feature dim: {loaded['feature_dim']}")
print(f"Method: {loaded['method']}")
print(f"\nSplits:")
print(f"  Train: {len(loaded['splits']['train'])}")
print(f"  Val: {len(loaded['splits']['val'])}")
print(f"  Test: {len(loaded['splits']['test'])}")

# Sample recording
sample_rec = list(loaded['data'].keys())[0]
sample_data = loaded['data'][sample_rec]
print(f"\nSample recording ({sample_rec}):")
print(f"  Embeddings shape: {sample_data['embeddings'].shape}")
print(f"  Labels: {sample_data['labels'][:5]}...")
print(f"  Descriptions: {sample_data['descriptions'][:2]}...")

## Done!

The step embeddings have been saved. You can now proceed to:
- **Step 2**: `extension_step2_verification_baseline.ipynb` - Train verification models
- **Step 3**: `extension_step3_task_graph_matching.ipynb` - Match steps to task graphs
- **Step 4**: `extension_step4_gnn_classification.ipynb` - GNN classification

For ActionFormer experiments and hyperparameter analysis, see:
- `extension_step1_actionformer.ipynb`